# Retrieval-Augmented Generation (RAG)

This ipython notebook will show how to implement and use Retrieval-Augmented Generation (RAG) for question answering tasks, with a focus on leveraging external knowledge sources to improve the performance of language models.


## Step 1: Install Required Libraries and load pages from the PDF files

In [ ]:
from pathlib import Path
from pypdf import PdfReader
import numpy as np
from dotenv import load_dotenv
from google import genai
import json
from google.genai import types
import time
import chromadb




In [ ]:
load_dotenv()
client = genai.Client()

In [ ]:
def load_pages(folder="data"):
    pages = []
    for pdf in sorted(Path(folder).glob("*.pdf")):
        reader = PdfReader(pdf)
        for i, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ""
            pages.append({"source": pdf.name, "page": i, "text": text})
    return pages

pages = load_pages()
print(len(pages), "pages")

## Step 2: Divide the obtained text into chunks of similar sizes

In [ ]:
def chunk_text(text, size=800, overlap=100):
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + size, n)
        if end < n:
            # avoid cutting in the middle of a word
            last_space = text.rfind(" ", start, end)
            if last_space > start + size // 2:
                end = last_space
        piece = text[start:end].strip()
        if piece:
            chunks.append(piece)
        if end >= n:
            break
        start = end - overlap
    return chunks


def build_chunks(pages, size=800, overlap=100, min_len=100):
    chunks = []
    for p in pages:
        for n, piece in enumerate(chunk_text(p["text"], size, overlap), start=1):
            if len(piece) < min_len:
                continue
            chunks.append({
                "id": f'{p["source"]}-p{p["page"]}-c{n}',
                "text": piece,
                "source": p["source"],
                "page": p["page"],
            })
    return chunks


chunks = build_chunks(pages)
print(len(chunks), "chunks")
assert len({c["id"] for c in chunks}) == len(chunks), "duplicate ids"

## Step 3: Sanity Check

Check if the generated chunks picked at random make sense to read, as well to check if the overlap is working.

In [ ]:
import random
for c in random.sample(chunks, 5):
    print(c["id"], len(c["text"]))
    print(c["text"], "\n---")

In [ ]:
for x, y in zip(chunks, chunks[1:]):
    if x["source"] == y["source"] and x["page"] == y["page"]:
        print(x["id"], y["id"])
        print(repr(x["text"][-120:]))
        print(repr(y["text"][:120]))
        print("overlap found:", y["text"][:60] in x["text"])
        break

## Step 4: Store the chunks in a JSON file


In [ ]:
with open("chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

## Step 5: Generate Embeddings and Store in ChromaDB

In [ ]:

EMBEDDING_MODEL = "gemini-embedding-001"

def get_embeddings(texts, task="RETRIEVAL_DOCUMENT") -> np.ndarray:
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=texts,
        config=types.EmbedContentConfig(task_type=task),
    )
    assert len(response.embeddings) == len(texts), "embedding count mismatch"
    return np.array([e.values for e in response.embeddings])

vecs = get_embeddings([c["text"] for c in chunks[:3]])
print(vecs.shape)   # (3, vector_length)

In [ ]:
db = chromadb.PersistentClient(path="chroma_db")
col = db.get_or_create_collection(name="docs", metadata={"hnsw:space": "cosine"})
print(col.count())

In [ ]:
def embed_and_store(chunks, col, batch_size=50, per_minute=90):
    done = set(col.get(ids=[c["id"] for c in chunks])["ids"])
    todo = [c for c in chunks if c["id"] not in done]
    print(f"{len(done)} already stored, {len(todo)} to embed")

    for i in range(0, len(todo), batch_size):
        batch = todo[i:i + batch_size]
        for attempt in range(4):
            try:
                vecs = get_embeddings([c["text"] for c in batch]).tolist()
                break
            except Exception as e:
                wait = 20 * (attempt + 1)
                print(f"failed: {str(e)[:100]} | retrying in {wait}s")
                time.sleep(wait)
        else:
            raise RuntimeError("batch failed after 4 attempts")

        col.upsert(
            ids=[c["id"] for c in batch],
            documents=[c["text"] for c in batch],
            embeddings=vecs,
            metadatas=[{"source": c["source"], "page": c["page"]} for c in batch],
        )
        print(f"stored {min(i + batch_size, len(todo))}/{len(todo)}")
        time.sleep(60 * len(batch) / per_minute)   # about 33s for 50 texts

embed_and_store(chunks, col)
print(col.count(), "items in collection")

# Step 6: Query Testing


In [47]:
def retrieve(question, k=4):
    qv = get_embeddings([question], task="RETRIEVAL_QUERY")[0].tolist()
    res = col.query(query_embeddings=[qv], n_results=k)
    return list(zip(res["documents"][0], res["metadatas"][0], res["distances"][0]))

for doc, meta, dist in retrieve("O que é uma ferida crónica?"):
    print(round(dist, 3), meta)
    print(doc[:200], "\n---")


#Questions asked
## What kind of political system does Portugal have?
## What is a chronic wound?
## Who is Ryze?
## How do I bake a cake?
## O que é uma ferida crónica?


0.265 {'source': 'Automated_Dressing_Saturation_Analysis.pdf', 'page': 17}
Chapter 1
Introduction
1.1 Context
A chronic wound is defined as a wound that does not follow a normal healing process, nor-
mally taking more than 4 to 12 weeks without reaching full anatomic and fun 
---
0.282 {'source': 'Automated_Dressing_Saturation_Analysis.pdf', 'page': 17}
ility
limitations and the risk of recurrent infections. Multiple observational studies have reported that
the presence of chronic wounds of various types is estimated at 2.21 per 1000 people [38], al- 
---
0.296 {'source': 'Automated_Dressing_Saturation_Analysis.pdf', 'page': 4}
Resumo
As feridas crónicas aplicam um peso significativo aos sistemas de saúde e à qualidade de vida dos
doentes. Durante o processo de cicatrização das feridas crónicas, o excesso de exsudado faz com 
---
0.318 {'page': 87, 'source': 'Automated_Dressing_Saturation_Analysis.pdf'}
cent Methods for
Non-Contact Chronic Wound Analysis”. In:Applied Sciences10.21 (202